# Cursul 2. Colectarea datelor prin API
**Fluxuri de lucru pentru știința datelor · Sociologie · 2026**

În C1 am citit fișiere și am ajuns la valori din liste și dicționare. Acum cerem date direct de la sursă, cu Python: de la **Banca Mondială (BM)** și de la **Eurostat**.

Pe lângă indicatorii despre locuri, aducem azi și **migrația**: câți oameni dintr-o țară trăiesc în alta. E a doua jumătate a ghicitorii din C1.

**Citește, anticipează, rulează, explică.**

| Pas | Ce facem | Timp |
|---|---|---|
| 1-2 | Pregătire; cum arată o cerere | 10 min |
| 3-5 | Banca Mondială: prima cerere, paginarea, datele proiectului | 35 min |
| 6-7 | Eurostat: documentația, prima cerere, apoi toate regiunile | 25 min |
| 8 | Migrația: un tabel cu două țări pe rând | 10 min |
| 9 | Echipele aleg ce date încearcă | 10 min |

## 1. Pregătire
Folosim același mediu ca în C1. `requests` trimite cereri pe internet, `pandas` face tabele. Amândouă sunt în `requirements.txt`, deci sunt deja instalate. Toate fișierele de azi merg în `data/c2`.

In [1]:
import requests
import pandas as pd
from pathlib import Path

# Folderul pentru fișierele de azi (dacă există deja, nu se întâmplă nimic)
Path("data/c2").mkdir(exist_ok=True)

## 2. Cum arată o cerere
**Notebook-ul trimite o cerere, serverul răspunde cu date.** Cererea e o adresă web:

`https://api.worldbank.org/v2/country/RO/indicator/SP.POP.TOTL?format=json&date=2022:2024`

| Parte | Ce înseamnă |
|---|---|
| `https://api.worldbank.org/v2` | serverul |
| `/country/RO/indicator/SP.POP.TOTL` | resursa: populația României |
| după `?` | parametrii: formatul și anii |

Deschide adresa în browser. **Anticipează:** câte valori o să vezi pentru trei ani?

## 3. Banca Mondială: prima cerere
Punem parametrii într-un dicționar, iar `requests` construiește adresa din ei.

In [2]:
url = "https://api.worldbank.org/v2/country/RO/indicator/SP.POP.TOTL"
parametri = {"format": "json", "date": "2022:2024"}

raspuns = requests.get(url, params=parametri)   # trimitem cererea
print(raspuns.url)           # adresa construită din url și parametri
print(raspuns.status_code)   # 200 = a mers

https://api.worldbank.org/v2/country/RO/indicator/SP.POP.TOTL?format=json&date=2022%3A2024
200


`200` înseamnă că serverul a răspuns. `404` ar fi o adresă greșită, `429` prea multe cereri. Acum citim conținutul ca JSON, adică liste și dicționare, ca în C1.

In [3]:
date = raspuns.json()   # textul răspunsului devine liste și dicționare
date

[{'page': 1,
  'pages': 1,
  'per_page': 50,
  'total': 3,
  'sourceid': '2',
  'lastupdated': '2026-10-08'},
 [{'indicator': {'id': 'SP.POP.TOTL', 'value': 'Population, total'},
   'country': {'id': 'RO', 'value': 'Romania'},
   'countryiso3code': 'ROU',
   'date': '2024',
   'value': 19051804,
   'unit': '',
   'obs_status': '',
   'decimal': 0},
  {'indicator': {'id': 'SP.POP.TOTL', 'value': 'Population, total'},
   'country': {'id': 'RO', 'value': 'Romania'},
   'countryiso3code': 'ROU',
   'date': '2023',
   'value': 19061062,
   'unit': '',
   'obs_status': '',
   'decimal': 0},
  {'indicator': {'id': 'SP.POP.TOTL', 'value': 'Population, total'},
   'country': {'id': 'RO', 'value': 'Romania'},
   'countryiso3code': 'ROU',
   'date': '2022',
   'value': 19048502,
   'unit': '',
   'obs_status': '',
   'decimal': 0}]]

### 3.1. Recunoaștem structura din C1
Răspunsul e o listă cu două elemente: `[0]` descrie răspunsul (câte valori sunt, pe câte pagini), `[1]` e lista valorilor.

In [4]:
meta = date[0]     # informații despre pagini
valori = date[1]   # lista valorilor

print(meta)
valori[0]

{'page': 1, 'pages': 1, 'per_page': 50, 'total': 3, 'sourceid': '2', 'lastupdated': '2026-10-08'}


{'indicator': {'id': 'SP.POP.TOTL', 'value': 'Population, total'},
 'country': {'id': 'RO', 'value': 'Romania'},
 'countryiso3code': 'ROU',
 'date': '2024',
 'value': 19051804,
 'unit': '',
 'obs_status': '',
 'decimal': 0}

In [ ]:
# Drumul până la o valoare, ca în C1:
# afișează numele țării, anul și valoarea primului element din valori

`pd.json_normalize` transformă lista de dicționare într-un tabel. Dicționarele din interior devin coloane cu punct în nume, de exemplu `country.id`.

In [ ]:
# Lista de dicționare devine tabel; păstrăm trei coloane
pd.json_normalize(valori)[["country.id", "date", "value"]]

**Exercițiu.** Fă aceeași cerere pentru Spania (`ES`). Ce se schimbă în adresă?

In [ ]:
# Aceeași cerere pentru Spania: schimbă doar codul țării din adresă

## 4. Paginarea: de ce nu vine tot răspunsul

Un API nu trimite un set mare dintr-o dată. Îl împarte în **pagini**, ca fiecare răspuns să rămână mic și rapid. La Banca Mondială, o pagină are implicit **50 de valori**.

Problema: serverul nu anunță că a trimis doar o parte. Codul e tot 200 și nu apare nicio eroare. Singurul semn e primul element al răspunsului, cel cu paginarea:

| Cheia | Ce spune |
|---|---|
| `total` | câte valori există în total |
| `per_page` | câte valori încap pe o pagină |
| `pages` | câte pagini sunt: `total ÷ per_page`, rotunjit în sus |
| `page` | ce pagină a venit acum |

**Regula:** dacă `pages` este 1, a venit tot. Dacă este mai mare, a venit doar prima pagină; celelalte trebuie cerute.

### 4.1. Ce vine fără `per_page`

Cerem populația României pentru 1960 la 2024, adică 65 de ani, fără să spunem câte valori vrem pe pagină.

**Anticipează:** câte valori o să vină?

In [ ]:
parametri = {"format": "json", "date": "1960:2024"}    # fără per_page

# răspunsul are două elemente; le despachetăm direct: paginarea și valorile
meta, valori = requests.get(url, params=parametri).json()

print("Există în total:", meta["total"], "valori")
print("Pe o pagină încap:", meta["per_page"])
print("Pagini:", meta["pages"])
print("Au venit:", len(valori), "valori, din", valori[0]["date"], "până în", valori[-1]["date"])

Au venit 50 de valori, din 2024 până în 1975. Codul a fost 200, deci cererea a reușit, și totuși **lipsesc 15 ani**, de la 1960 la 1974. Ei stau pe pagina 2 și vin doar dacă îi cerem.

### 4.2. A doua pagină

Aceeași cerere, cu un singur parametru în plus: `page`.

`{**parametri, "page": 2}` înseamnă: toate cheile din `parametri`, plus `page` cu valoarea 2. Dicționarul `parametri` rămâne neschimbat.

**Exercițiu.** Cere pagina 2. Câte valori vin și din ce ani? Câte sunt, împreună cu cele de pe pagina 1?

In [ ]:
# Cere pagina 2: aceeași cerere, cu params={**parametri, "page": 2}
# despachetează răspunsul în meta2 și valori2, ca mai sus
# afișează câte valori au venit, anul primei și al ultimei valori
# și câte sunt împreună cu cele de pe pagina 1

### 4.3. O funcție care aduce toate paginile

Am putea cere pagini foarte mari, de exemplu `"per_page": 1000`. Dar câte valori încap depinde de cerere: pentru cele 37 de țări ale proiectului și 65 de ani sunt 37 × 65 = 2.405 valori, deci tot 3 pagini. Mai sigur e o funcție care cere singură paginile, oricâte ar fi.

Funcția face, pe rând, ce am făcut mai sus de mână:

1. cere pagina 1;
2. adaugă valorile ei la o listă;
3. dacă pagina era ultima (`pagina` a ajuns la `pages`), se oprește și dă lista;
4. altfel, cere pagina următoare și reia de la pasul 2.

In [6]:
def cere_tot(url, parametri):
    """Toate valorile unei cereri la Banca Mondială, oricâte pagini ar avea."""

    # 1. O primă cerere, doar ca să aflăm câte pagini sunt.
    #    Din răspuns luăm numai prima parte, [0], cea cu paginarea.
    meta = requests.get(url, params=parametri).json()[0]

    # 2. Cerem fiecare pagină, de la 1 până la ultima.
    #    Din fiecare luăm numai a doua parte, [1], cea cu valorile.
    toate = []
    for pagina in range(1, meta["pages"] + 1):
        valori = requests.get(url, params={**parametri, "page": pagina}).json()[1]
        toate = toate + valori          # lipim valorile paginii la cele de până acum
        print("pagina", pagina, "din", meta["pages"])

    # 3. Dăm înapoi toate valorile, din toate paginile.
    return toate

O încercăm cu pagini mici, de câte 20, ca să se vadă cum lucrează.

**Anticipează:** câte cereri trimite funcția pentru cele 65 de valori?

In [ ]:
valori = cere_tot(url, {"format": "json", "date": "1960:2024", "per_page": 20})
len(valori)

Funcția nu se schimbă: dacă datele încap într-o pagină, face o cerere; dacă nu, le cere și pe celelalte.

**Exercițiu.** Aceeași funcție, pentru opt țări: România, Italia, Spania, Germania, Franța, Ungaria, Bulgaria și Austria, cu câte 100 de valori pe pagină. Câte pagini sunt? Câte valori vin în total?

In [ ]:
# Țările intră în adresă despărțite prin ";"
url_8 = "https://api.worldbank.org/v2/country/RO;IT;ES;DE;FR;HU;BG;AT/indicator/SP.POP.TOTL"

# Cere toate valorile cu cere_tot, pentru anii 1960:2024, câte 100 pe pagină
# apoi afișează câte valori au venit în total

### 4.4. Funcția pentru proiect

`cere_bm` cere pagini mari, de câte 1000, ca să fie cât mai puține cereri, iar `cere_tot` le aduce pe toate, oricâte ar fi. La final, lista devine tabel.

In [7]:
def cere_bm(indicator, tari, ani="2022:2024"):
    # Țările intră în adresă despărțite prin ";", de exemplu RO;IT;ES
    url = "https://api.worldbank.org/v2/country/" + ";".join(tari) + "/indicator/" + indicator
    valori = cere_tot(url, {"format": "json", "date": ani, "per_page": 1000})

    tabel = pd.json_normalize(valori)[["country.id", "date", "value"]]   # păstrăm trei coloane
    tabel.columns = ["tara", "an", "valoare"]                            # le dăm nume românești
    return tabel

## 5. Contextul național pentru proiect
Țările de mai jos sunt cele care apar în fișierele SCI regionale ale proiectului. Cerem trei indicatori pentru fiecare.

Atenție la coduri: Banca Mondială și SCI scriu Grecia `GR`. Eurostat o scrie `EL`. Ne întoarcem la asta în C4.

In [ ]:
TARI = ["AL", "AT", "BA", "BE", "BG", "CH", "CY", "CZ", "DE", "DK", "EE", "ES", "FI",
        "FR", "GR", "HR", "HU", "IE", "IS", "IT", "LT", "LU", "LV", "ME", "MK", "MT",
        "NL", "NO", "PL", "PT", "RO", "RS", "SE", "SI", "SK", "TR", "XK"]

# codul indicatorului la Banca Mondială -> numele scurt pe care îl folosim noi
INDICATORI_BM = {"SP.POP.TOTL": "populatie",
                 "NY.GDP.PCAP.CD": "pib_locuitor_usd",
                 "IT.NET.USER.ZS": "internet_pct"}

In [ ]:
tabele = []
for cod, nume in INDICATORI_BM.items():
    tabel = cere_bm(cod, TARI)
    tabel["indicator"] = nume    # notăm ce indicator e în tabel
    tabele.append(tabel)

bm = pd.concat(tabele)           # lipim cele trei tabele unul sub altul
bm = bm[["country.id", "date", "indicator", "value"]]
bm.columns = ["tara", "an", "indicator", "valoare"]
bm.to_csv("data/c2/bm.csv", index=False)
bm.head()

## 6. Eurostat: date pe regiuni
Eurostat are date pe regiuni: țări, regiuni mari (NUTS 2) și județe (NUTS 3). Ca la Banca Mondială, pornim de la documentație: [Eurostat: primii pași cu API-ul](https://ec.europa.eu/eurostat/web/user-guides/data-browser/api-data-access/api-getting-started/api).

O cerere are aceeași structură: serverul, apoi ce cerem, apoi filtrele, după `?`. Densitatea populației în Cluj, în 2024:

`https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/demo_r_d3dens?geo=RO113&time=2024`

| Parte | Ce înseamnă |
|---|---|
| `https://ec.europa.eu/eurostat/api/dissemination` | serverul |
| `/statistics/1.0/data/` | serviciul de statistici, versiunea 1.0: date |
| `demo_r_d3dens` | setul de date: densitatea populației pe regiuni NUTS 3 |
| după `?` | filtrele: regiunea (`geo=RO113`, Cluj) și anul (`time=2024`) |

Deschide adresa în browser. **Anticipează:** câte valori o să vezi?

### 6.1. Ce seturi există
Codul unui set, de exemplu `demo_r_d3dens`, se vede în [Data Browser](https://ec.europa.eu/eurostat/databrowser/), lângă titlul tabelului. Lista completă e în documentație, la [cuprinsul bazei de date](https://ec.europa.eu/eurostat/web/user-guides/data-browser/api-data-access/api-detailed-guidelines/catalogue-api/toc): un fișier text cu toate seturile, câte unul pe rând.

Coloanele fișierului sunt despărțite prin tab (`\t`), deci îl citim cu `pd.read_csv` și `sep="\t"`, direct de la adresă.

In [ ]:
CUPRINS = "https://ec.europa.eu/eurostat/api/dissemination/catalogue/toc/txt?lang=en"

cuprins = pd.read_csv(CUPRINS, sep="\t")     # coloanele sunt despărțite prin tab
cuprins.head()

Cuprinsul are și foldere (capitolele) și seturi de date. Păstrăm doar seturile, fiecare o singură dată.

In [ ]:
seturi = cuprins[cuprins["type"] == "dataset"].drop_duplicates("code")
len(seturi)

Căutăm un cuvânt în titlu; titlurile sunt în engleză. **Anticipează:** câte seturi sunt pe județe, cu „NUTS 3” în titlu?

In [ ]:
nuts3 = seturi[seturi["title"].str.contains("NUTS 3")]
len(nuts3)

Primele zece, cu codul, titlul și ultimul an cu date (`data end`). Anul din cerere trebuie să fie cel mult `data end`.

In [ ]:
nuts3[["code", "title", "data end"]].head(10)

**Exercițiu.** Caută seturile despre cetățenie: cuvântul `citizenship`.

In [ ]:
# Aceeași căutare, cu "citizenship" în loc de "NUTS 3"

### 6.2. Prima cerere din Python
Aceeași adresă, cu `requests`: adresa de bază plus codul setului, iar filtrele în dicționar, ca la Banca Mondială.

In [ ]:
EUROSTAT = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/"

raspuns = requests.get(EUROSTAT + "demo_r_d3dens", params={"geo": "RO113", "time": 2024})
print(raspuns.url)           # adresa construită, aceeași ca în browser
print(raspuns.status_code)   # 200 = a mers

Și aici răspunsul e JSON, dar altfel organizat decât la Banca Mondială: titlul setului, numele fiecărui cod și, separat, valorile.

In [ ]:
date = raspuns.json()

print(date["label"])                                   # titlul setului
print(date["dimension"]["geo"]["category"]["label"])   # regiunea cerută, cu numele ei
print(date["value"])                                   # valoarea, cu numărul ei: 0

O singură valoare, cu numărul 0. Când cerem mai multe regiuni, valorile vin numerotate: 0, 1, 2... Le potrivim cu regiunile la pasul următor.

### 6.3. Două regiuni: bilețelele de la garderobă
Cerem Cluj și Bihor (`RO111`) deodată: o listă în `geo` cere mai multe regiuni.

Răspunsul are două dicționare, ca la o garderobă:
- `numere`: fiecare regiune are un bilețel cu un număr;
- `valori`: la fiecare număr stă o valoare, ca haina în cuier.

In [ ]:
date = requests.get(EUROSTAT + "demo_r_d3dens", params={"geo": ["RO113", "RO111"], "time": 2024}).json()

numere = date["dimension"]["geo"]["category"]["index"]    # ce număr are fiecare regiune
valori = date["value"]                                     # ce valoare stă la fiecare număr
print(numere)
print(valori)

Bihor are numărul 0, Cluj numărul 1, deși Clujul a fost cerut primul: numerele le dă Eurostat, nu ordinea din cerere. La numărul 0 stă 74,2, la numărul 1 stă 104,8.

Potrivim de mână, pentru Cluj: luăm numărul lui, apoi valoarea de la acel număr. În `valori`, numerele sunt scrise ca text (`"0"`, `"1"`), de aceea `str(numar)`.

In [ ]:
numar = numere["RO113"]     # numărul Clujului: 1
valori[str(numar)]          # valoarea de la numărul 1

Pentru toate regiunile, aceeași potrivire, într-o buclă. `.get` dă `None` dacă la un număr nu stă nicio valoare: regiunea nu are dată, ceea ce nu înseamnă zero.

In [ ]:
for regiune, numar in numere.items():
    print(regiune, valori.get(str(numar)))

### 6.4. O funcție care face tabelul
`cere_eurostat` face exact pașii de mai sus: cererea, cele două dicționare, bucla. În loc să afișeze, adună rândurile într-un tabel. O scriem o dată, apoi o folosim ca pe `cere_bm`.

Regula: într-o cerere variază un singur lucru, regiunea; celelalte filtre (anul, unitatea, sexul...) au câte o singură valoare. Dacă vin mai multe valori decât regiuni, înseamnă că variază și altceva, de exemplu anul uitat, iar funcția se oprește.

`pe` spune ce stă pe rândurile tabelului: implicit regiunea (`geo`). Îl schimbăm o singură dată, la migrație.

In [ ]:
def cere_eurostat(dataset, parametri, pe="geo"):
    """Un set Eurostat, ca tabel: un rând pentru fiecare regiune."""
    date = requests.get(EUROSTAT + dataset, params=parametri).json()

    numere = date["dimension"][pe]["category"]["index"]   # bilețelele: ce număr are fiecare regiune
    valori = date["value"]                                 # cuierul: ce valoare stă la fiecare număr

    if len(valori) > len(numere):                          # mai multe valori decât regiuni: variază și altceva
        raise ValueError("Mai fixează un filtru, de exemplu anul")

    tabel = []
    for regiune, numar in numere.items():                  # pentru fiecare regiune, valoarea de la numărul ei
        tabel.append({pe: regiune, "valoare": valori.get(str(numar))})
    return pd.DataFrame(tabel)

Aceeași cerere, acum ca tabel: un rând pe regiune.

In [ ]:
cere_eurostat("demo_r_d3dens", {"geo": ["RO113", "RO111"], "time": 2024})

**Exercițiu.** Aceeași cerere, pentru Timiș (`RO424`).

In [ ]:
# Același set și același an; în "geo" pune codul Timișului, "RO424"

## 7. Toate regiunile Europei
În loc de `geo` punem `geoLevel`: `nuts3` aduce toate județele Europei (și echivalentele lor), `nuts2` toate regiunile mari.

Luăm indicatorii pe rând, fiecare în fișierul lui. Îi punem împreună mai târziu, la agregare.

### 7.1. Densitatea populației
Setul `demo_r_d3dens`, persoane pe km² (sursa: [Data Browser](https://ec.europa.eu/eurostat/databrowser/view/demo_r_d3dens/default/table)). Întâi județele:

In [ ]:
densitate_nuts3 = cere_eurostat("demo_r_d3dens", {"geoLevel": "nuts3", "time": 2024})
densitate_nuts3.head()

**Interpretează:** câte regiuni au venit și câte au valoare? O regiune fără valoare nu are zero locuitori: sursa nu are dată pentru anul cerut.

In [ ]:
print(len(densitate_nuts3), "regiuni")
print(densitate_nuts3["valoare"].count(), "cu valoare")

Salvăm județele, apoi facem același lucru pentru regiunile mari.

In [ ]:
densitate_nuts3.to_csv("data/c2/densitate_nuts3.csv", index=False)

In [ ]:
densitate_nuts2 = cere_eurostat("demo_r_d3dens", {"geoLevel": "nuts2", "time": 2024})
densitate_nuts2.to_csv("data/c2/densitate_nuts2.csv", index=False)

### 7.2. Populația
Setul `demo_r_pjangrp3` (sursa: [Data Browser](https://ec.europa.eu/eurostat/databrowser/view/demo_r_pjangrp3/default/table)). Are mai multe dimensiuni, deci fixăm mai multe filtre: `sex=T` și `age=TOTAL` înseamnă toată populația, nu o grupă; `unit=NR`, număr de persoane.

In [ ]:
populatie_nuts3 = cere_eurostat("demo_r_pjangrp3", {"geoLevel": "nuts3", "sex": "T", "age": "TOTAL", "unit": "NR", "time": 2024})
populatie_nuts3.to_csv("data/c2/populatie_nuts3.csv", index=False)

In [ ]:
populatie_nuts2 = cere_eurostat("demo_r_pjangrp3", {"geoLevel": "nuts2", "sex": "T", "age": "TOTAL", "unit": "NR", "time": 2024})
populatie_nuts2.to_csv("data/c2/populatie_nuts2.csv", index=False)

### 7.3. PIB pe locuitor
Setul `nama_10r_3gdp` (sursa: [Data Browser](https://ec.europa.eu/eurostat/databrowser/view/nama_10r_3gdp/default/table)). `unit=PPS_EU27_2020_HAB`: PIB pe locuitor la paritatea puterii de cumpărare, ca valorile să se poată compara între țări.

In [ ]:
pib_nuts3 = cere_eurostat("nama_10r_3gdp", {"geoLevel": "nuts3", "unit": "PPS_EU27_2020_HAB", "time": 2024})
pib_nuts3.to_csv("data/c2/pib_locuitor_nuts3.csv", index=False)

In [ ]:
pib_nuts2 = cere_eurostat("nama_10r_3gdp", {"geoLevel": "nuts2", "unit": "PPS_EU27_2020_HAB", "time": 2024})
pib_nuts2.to_csv("data/c2/pib_locuitor_nuts2.csv", index=False)

## 8. Migrația: două țări pe un rând
Setul `migr_pop1ctz` (sursa: [Data Browser](https://ec.europa.eu/eurostat/databrowser/view/migr_pop1ctz/default/table)) spune câți oameni cu o anumită **cetățenie** (`citizen`) trăiesc într-o țară (`geo`), la 1 ianuarie.

### 8.1. Unde trăiesc cetățenii români
E ghicitoarea din C1. Filtrele comune le punem o dată, într-un dicționar:

In [ ]:
filtre = {"sex": "T", "age": "TOTAL", "unit": "NR", "time": 2024}     # toată populația, în persoane, în 2024

Fixăm și cetățenia, `"citizen": "RO"`; variază doar țara. `{**filtre, "citizen": "RO"}` înseamnă: filtrele comune, plus cetățenia, ca la paginare.

In [ ]:
romani = cere_eurostat("migr_pop1ctz", {**filtre, "citizen": "RO"})
romani.head()

Ordonăm de la cea mai mare valoare la cea mai mică.

In [ ]:
romani.sort_values("valoare", ascending=False).head(6)

Primul rând e chiar România: cetățenii români care trăiesc în țară. Urmează Italia, Germania și Spania. Italia are cei mai mulți români din UE, dar în SCI-ul României era abia pe locul 6. Migrația și legătura socială nu sunt același lucru; de ce diferă aflăm în cursurile următoare.

**Exercițiu.** Unde trăiesc cei mai mulți cetățeni italieni? Schimbă doar codul cetățeniei.

In [ ]:
# Aceeași cerere ca pentru români, cu "IT" în loc de "RO", apoi ordonarea

### 8.2. Tabelul pentru proiect: toate perechile
Pentru proiect vrem toate perechile: fiecare țară, cu fiecare cetățenie. Ca la SCI, un rând descrie **o relație între două țări**. Migrația are însă **sens**: românii din Italia nu sunt italienii din România; la SCI, RO-IT și IT-RO au aceeași valoare.

Într-o cerere variază un singur lucru, deci cerem țară cu țară. Întâi lista țărilor: cerem totalul locuitorilor (`citizen=TOTAL`), un rând pe țară.

In [ ]:
tari_eu = cere_eurostat("migr_pop1ctz", {**filtre, "citizen": "TOTAL"})["geo"]
list(tari_eu)

O țară, ca exemplu: câți locuitori are Italia din fiecare cetățenie. Acum fixăm țara și variază cetățenia; `pe="citizen"` îi spune funcției că pe rânduri stau cetățeniile.

In [ ]:
italia = cere_eurostat("migr_pop1ctz", {**filtre, "geo": "IT"}, pe="citizen")
italia.head()

Același lucru pentru fiecare țară, într-o buclă; notăm țara în fiecare tabel și le lipim la final, ca la Banca Mondială. Sunt 45 de cereri, deci durează cam un minut.

In [ ]:
tabele = []
for tara in tari_eu:
    tabel = cere_eurostat("migr_pop1ctz", {**filtre, "geo": tara}, pe="citizen")   # un rând pe cetățenie
    tabel["geo"] = tara                                                             # notăm țara
    tabele.append(tabel)

migratie = pd.concat(tabele)     # lipim tabelele unul sub altul
migratie.shape

In [ ]:
migratie.to_csv("data/c2/migratie.csv", index=False)

## 9. Echipele: ce date încercați
Alegeți o direcție provizorie și verificați că datele se pot lua. Întrebarea de cercetare o scrieți în C3.

| Direcție | Ce încercați | Set Eurostat | Filtre | Sursa |
|---|---|---|---|---|
| Migrație | alte cetățenii sau alt an | `migr_pop1ctz` | `sex=T, age=TOTAL, unit=NR` | [Data Browser](https://ec.europa.eu/eurostat/databrowser/view/migr_pop1ctz/default/table) |
| Demografie | ponderea persoanelor de 65+ ani (NUTS3) | `demo_r_pjanind3` | `indic_de=PC_Y65_MAX, unit=PC` | [Data Browser](https://ec.europa.eu/eurostat/databrowser/view/demo_r_pjanind3/default/table) |
| Educație | studii superioare, 25-64 ani (NUTS2) | `edat_lfse_04` | `isced11=ED5-8, age=Y25-64, sex=T, unit=PC` | [Data Browser](https://ec.europa.eu/eurostat/databrowser/view/edat_lfse_04/default/table) |
| Piața muncii | rata șomajului (NUTS2) | `lfst_r_lfu3rt` | `isced11=TOTAL, age=Y15-74, sex=T, unit=PC` | [Data Browser](https://ec.europa.eu/eurostat/databrowser/view/lfst_r_lfu3rt/default/table) |

La fiecare adăugați și anul, de exemplu `time=2024`; ultimul an cu date îl vedeți în cuprins, la `data end`. Dacă lipsește un filtru, `cere_eurostat` se oprește și vă cere să-l adăugați. Alt indicator îl găsiți căutând în cuprins, ca la 6.1, sau în Data Browser.

**Exercițiul echipei**, în trei pași.

In [ ]:
# 1. Cererea pentru un singur județ: cere_eurostat, cu "geo" și filtrele din tabel

In [ ]:
# 2. Cererea pentru tot nivelul: aceleași filtre, cu "geoLevel" în loc de "geo"

In [ ]:
# 3. Salvarea tabelului în data/c2, cu to_csv

**Fișa echipei**

- Direcția: ...
- Indicatorul încercat (set și filtre): ...
- Câte regiuni au valoare: ...

## Ce ducem mai departe
În `data/c2` avem acum `bm.csv`, câte un fișier pe indicator și nivel, și `migratie.csv`. Sursa fiecăruia se vede în celula care l-a produs.

**Poți explica fără cod?** Ce e un parametru? Cum afli dacă ai primit toate paginile? Unde găsești codul unui set Eurostat? De ce o valoare lipsă nu e zero? De ce migrația are sens, iar SCI nu?

### Documentație
- [Banca Mondială: structura cererilor](https://datahelpdesk.worldbank.org/knowledgebase/articles/898581-api-basic-call-structures)
- [Eurostat: primii pași cu API-ul](https://ec.europa.eu/eurostat/web/user-guides/data-browser/api-data-access/api-getting-started/api)
- [Eurostat: cuprinsul bazei de date](https://ec.europa.eu/eurostat/web/user-guides/data-browser/api-data-access/api-detailed-guidelines/catalogue-api/toc)
- [JSON-stat, formatul răspunsurilor Eurostat](https://json-stat.org/format/)